In [1]:
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# ===== choose dataset =====
dataset_name = 'MNIST'

# ===== DNN model =====
class CustomDNN(nn.Module):
    def __init__(self):
        super(CustomDNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# ===== load dataset =====
def load_dataset(name):
    transform = transforms.Compose([
        transforms.Grayscale(),  # Ensure single channel
        transforms.ToTensor(),
    ])
    if name == 'MNIST':
        dataset = torchvision.datasets.MNIST
    elif name == 'FashionMNIST':
        dataset = torchvision.datasets.FashionMNIST
    elif name == 'KMNIST':
        dataset = torchvision.datasets.KMNIST
    else:
        raise ValueError("Invalid dataset name!")

    train_data = dataset(root='./data', train=True, transform=transform, download=True)
    test_data = dataset(root='./data', train=False, transform=transform, download=True)

    return train_data, test_data

# ===== train and test =====
def train_and_test():
    train_dataset, test_dataset = load_dataset(dataset_name)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CustomDNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

    # accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Accuracy: {100 * correct / total:.2f}%")

# ===== main =====
if __name__ == '__main__':
    train_and_test()


100%|██████████| 9.91M/9.91M [00:00<00:00, 15.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 482kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.16MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.24MB/s]


Epoch 1, Loss: 0.2445
Epoch 2, Loss: 0.0924
Epoch 3, Loss: 0.0628
Epoch 4, Loss: 0.0465
Epoch 5, Loss: 0.0382
Epoch 6, Loss: 0.0321
Epoch 7, Loss: 0.0270
Epoch 8, Loss: 0.0235
Epoch 9, Loss: 0.0206
Epoch 10, Loss: 0.0158
Accuracy: 98.06%
